<!-- your logo here! -->
<img src="https://ac.auriasolutions.com/library/images/auria_logo_sm.jpg"/>
<img src="https://github.com/bionadmin/distrib/raw/main/core.png"/>





# Attempt to update the Silver schema without losing data.

#### Debug options

In [1]:
debug_mode = False # Include debug select and print statements
force_remount = False #Normally mount points are left alone if they exists. Setting this to true will replace the mountpoints.
force_rebuild_of_all_tables = False

StatementMeta(, 4e9b6787-4649-41c6-979d-744279977fa4, 3, Finished, Available, Finished, False)

#### Some items we need...

In [2]:
from datetime import datetime
now = datetime.now()
tag = now.strftime("%Y%m%d%H%M%S")
print(tag)

useManagedTables = True
unmanagedArchiveCopies = 1
managedTablesKeepOldTableVersions = False
rebuildTableWhenColumnOrderChanges = True

destination_database_name="LH_BI_SILVER_TIER2"
spark.conf.set("sess.NB_SIL_Default_Table_destination_database_name", destination_database_name)



dest_base_path= "<Dest Folder>"

dest_additional_path="Silver/"
dest_base_path=dest_base_path.replace("<Dest Folder>",dest_additional_path)
destination_file_folder=""#set to None if no subfolder is desired. Include only internal slashes in the path.
if destination_file_folder==None or destination_file_folder=="":
    destination_database_default_path= dest_base_path
else:
    destination_database_default_path= dest_base_path + "/"+destination_file_folder
destination_filepath=destination_database_default_path+"/"

spark.conf.set("sess.NB_SIL_Default_Table_destination_database_default_path", destination_database_default_path)

StatementMeta(, 4e9b6787-4649-41c6-979d-744279977fa4, 4, Finished, Available, Finished, False)

20260407170158


#### Misc. Spark Options
If generating code, edit these in the template settings for consistency<br>
across all notebooks.

In [3]:
spark.conf.set("spark.sql.parquet.vorder.enabled","false")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled","true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize","1073741824")

StatementMeta(, 4e9b6787-4649-41c6-979d-744279977fa4, 5, Finished, Available, Finished, False)

In [4]:
%%sql
SET ANSI_MODE = TRUE

StatementMeta(, 4e9b6787-4649-41c6-979d-744279977fa4, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [5]:
def TableArraysAreEqual(old, new):
    if (len(old)!=len(new)):
        return False
    outerIndex=0
    for rowNew in new:
        innerIndex=0
        found = False
        for rowOld in old:
            if rowNew["col_name"]==rowOld["col_name"] and rowNew["data_type"]==rowOld["data_type"] and (rowNew["comment"] or "")==(rowOld["comment"] or ""):
                found=True
                break
            innerIndex=innerIndex+1
        if not found:
            if debug_mode:
                    print("Table structure has changed. Rebuilding table.")
            return False
        if rebuildTableWhenColumnOrderChanges and innerIndex!=outerIndex:
            if debug_mode:
                    print("Column Order has changed. Rebuilding table.")
            return False  
        outerIndex=outerIndex+1
    return True

def GetColumnByComment(arr,comment):
    if comment == "" or comment is None:
        return None
    for row in arr:
        if (row["comment"] is not None):
            if (comment in row["comment"]):
                return row
    return None

def GetColumnByName(arr, name):
    for row in arr:
        if (name == row["col_name"]):
            return row
    return None

def GetColumnByCommentThenName(arr, comment, name):
    row = GetColumnByComment(arr,comment)
    if row is None:
        row = GetColumnByName(arr,name)
    return row

def GenerateTableCreate(new, databaseName, tableName, partitions,location):

    stmt = "CREATE OR REPLACE TABLE "+databaseName+"."+tableName
    stmt=stmt+"\r\n("
    for row in new:
        stmt = stmt+"\r\n\t`"+row["col_name"]+"` "+row["data_type"]
        if row["comment"]:
            stmt = stmt +" COMMENT \""+row["comment"]+"\","
        else:
            stmt = stmt +","
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\n)"  
    stmt=stmt+"\r\nUSING DELTA"
    if partitions is not None:
        if len(partitions)>0:
            stmt=stmt+"\r\nPARTITIONED BY("
            firstPartition=True
            for par in partitions:
                if not firstPartition:
                    stmt+=","
                firstPartition=False
                stmt+="`"+par+"`"
            stmt=stmt+")"
    if not useManagedTables:
        stmt=stmt+"\r\nLOCATION \""+location+"\""
    if debug_mode:
        print (stmt)
    return stmt

def GenerateTableCopy(old, new, databaseName, oldTableName, newTableName):
    stmt = "INSERT INTO "+databaseName+"."+newTableName
    stmt=stmt+"\r\n("
    for row in new:
        stmt = stmt+"\r\n\t`"+row["col_name"]+"`,"
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\n)"   
    stmt=stmt+"SELECT "
    for row in new:
        oldRow = GetColumnByCommentThenName(old,row["comment"],row["col_name"])
        if oldRow is None:
            stmt=stmt+"\r\n\tNULL AS `"+row["col_name"]+"`,"
        else:
            stmt=stmt+"\r\n\tCAST(`"+oldRow["col_name"]+"` AS "+row["data_type"]+") AS `"+row["col_name"]+"`,"
    stmt=stmt[0:len(stmt)-1]#get rid of last trailing comma
    stmt=stmt+"\r\nFROM "+databaseName+"."+oldTableName
    if debug_mode:
        print (stmt)
    return stmt
 
def GenerateTableRename(databaseName, oldTableName, newTableName):
    stmt = "ALTER TABLE "+databaseName+"."+oldTableName +" RENAME TO "+newTableName
    if debug_mode:
        print (stmt)
    return stmt



def RemoveNonColumns(arr):
    returnArr = []
    for row in arr:
        if row["col_name"] == "# Partitioning":#Old way
            break  # we've gone past the columns. all done
        if row["col_name"] == "# Partition Information":#New Way
            break  # we've gone past the columns. all done
        if row["col_name"] == "# col_name":#New Way
            break  # we've gone past the columns. all done
        if row["data_type"] != "" and row["data_type"] is not None:
            returnArr.append(row)
    return returnArr
  
def TableExists(databaseName, tableName):
    table_list=spark.sql("show tables in "+databaseName)
    table_name=table_list.filter(table_list.tableName==tableName.lower()).collect()
    if table_name:
        return True
    else:
        return False
    
def UnmanagedRenameFolder(oldlocation,newlocation):
    if useManagedTables:
        return
    notebookutils.fs.mv(oldlocation, newlocation, True)

def UnmanagedDropTable(databaseName, tableName):
    spark.sql("DROP TABLE "+databaseName+"."+tableName)

def ManagedDropTable(databaseName, tableName):
    spark.sql("DROP TABLE "+databaseName+"."+tableName)
    
def UpdateTableStructure(old, new, databaseName, tableName, partitions, location, force):
    
    if (not force) and TableArraysAreEqual(old, new):
        if debug_mode:
            print ("Tables are the same. No need to update.")
        return
    stmt = GenerateTableCreate(new,databaseName, tableName+"_"+tag+"_TMP", partitions,location+"_"+tag+"_TMP")
    spark.sql(stmt)
    stmt = GenerateTableCopy(old,new,databaseName, tableName,tableName+"_"+tag+"_TMP")
    spark.sql(stmt)
    if useManagedTables:
        #With managed tables, the folder gets renamed with the table so this is all we need. Nice and simple.
        stmt = GenerateTableRename(databaseName,tableName, tableName+"_"+tag)
        spark.sql(stmt)
        stmt = GenerateTableRename(databaseName, tableName+"_"+tag+"_TMP",tableName)
        spark.sql(stmt)
        if not managedTablesKeepOldTableVersions:
            ManagedDropTable(databaseName, tableName+"_"+tag)
    if not useManagedTables:
        #with unmanaged, we need to rename the folder too so we don't have the new table's data sitting in a folder with a weird name...
        #basically since these are unmanaged we move the data around and then recreate the table which just updates the meta-data
        UnmanagedDropTable(databaseName,tableName)#drop our two tables.
        UnmanagedDropTable(databaseName,tableName+"_"+tag+"_TMP")#drop our two tables.
        UnmanagedRenameFolder(location,location+"_"+tag+"_PreviousVersion")#move the old version of the table's data out to a previous version folder.
        UnmanagedRenameFolder(location+"_"+tag+"_TMP",location)#move the new tables data into the folder with the basic table name
        stmt=GenerateTableCreate(new,databaseName, tableName, partitions,location)#create the table against the new path.
        if debug_mode:
            print(stmt)
        spark.sql(stmt)
        
def GetPartitions(collectedDescription):
    returnArr = []
    columns = RemoveNonColumns(collectedDescription)
    foundPartitionMarker = False
    newWay = False
    for row in collectedDescription:
        if row["col_name"] == "# Partitioning":  #older way
            foundPartitionMarker = True
            continue
        if row["col_name"] == "# Partition Information":  # newer way
            foundPartitionMarker = True
            newWay=True
            continue
        else:
            if not foundPartitionMarker:
                continue
        # if we made it this far, we are on the partitions section of the description.
        if row["col_name"] == "Not partitioned":
            return returnArr
        if newWay:
            potentialName = row["col_name"]
            if not potentialName.startswith("#"):
                for col in columns:
                    # confirm we are looking at a partition column
                    if col["col_name"].lower() == potentialName.lower():
                        returnArr.append(potentialName)
                        break
        else:
            if row["col_name"].startswith("Part "):  # partitions show up this way.
                potentialName = row[
                    "data_type"
                ]  # partition column names are in the type column.
                for col in columns:
                    # confirm we are looking at a partition column
                    if col["col_name"].lower() == potentialName.lower():
                        returnArr.append(potentialName)
                        break
    return returnArr


def StringArraysHaveSameElementsCI(one,two):
    if len(one)!=len(two):
        return False
    for strone in one:
        found = False
        for strtwo in two:
            if strone.lower()==strtwo.lower():
                found=True
                break
        if not found:
            return False
    return True
    
def RemoveOldArchiveCopies(tableName, filePath):
    if not useManagedTables:
        if unmanagedArchiveCopies>-1:#-1 means keep all copies!
            directoriesToParse=[]
            for fileinfo in notebookutils.fs.ls(filePath):
                name = fileinfo.name
                if name.startswith(tableName) and len(name)==len(tableName)+32 and name.endswith("_PreviousVersion/"):
                    if debug_mode:
                        print (fileinfo.path)
                    directoriesToParse.append(fileinfo.path)
            directoriesToParse.sort(reverse=True)
            filesToKeep = unmanagedArchiveCopies
            for directory in directoriesToParse:
                if filesToKeep>0:
                    filesToKeep=filesToKeep-1
                    continue
                else:
                    if debug_mode:
                        print("Removing archive directory "+directory)
                    notebookutils.fs.rm(directory,True)
                    
def TableIsManaged(databaseName, tableName):
    df = spark.sql("DESCRIBE EXTENDED "+databaseName+"."+tableName)
    arr = df.collect()
    foundDetailed = False
    for row in arr:
        if row["col_name"]=="# Detailed Table Information":
            foundDetailed=True
            continue
        if not foundDetailed:
            continue
        if row["col_name"]=="Type":
            if row["data_type"]=="MANAGED":
                return True
    return False

def ProcessTable(databaseName, tableName, comparisonArray, partitions, path):
    existingTableIsManaged=False
    tableExists = TableExists(databaseName,tableName)
    if tableExists:
        if debug_mode:
            print ("Table Exists")
        description = spark.sql("DESCRIBE "+databaseName+"."+tableName)
        oldpartitions = GetPartitions(description.collect())
        force = not StringArraysHaveSameElementsCI(oldpartitions,partitions)#rewrite table if partitions changed
        if force_rebuild_of_all_tables:
            force = True
        existingTableIsManaged = TableIsManaged(databaseName,tableName)
        if not force:  
            if (useManagedTables and not existingTableIsManaged) or (not useManagedTables and existingTableIsManaged):
                force = True
                if debug_mode:
                    if useManagedTables:
                        print("Converting tables from unmanaged to managed...")
                    else:
                        print("Converting tables from managed to unmanaged...")
        cols = RemoveNonColumns(description.collect())
        UpdateTableStructure(cols, comparisonArray, databaseName,tableName,partitions,path,force)
    else:
        if debug_mode:
            print ("Table Does not Exist. Creating Table...")
        stmt = GenerateTableCreate(comparisonArray,databaseName,tableName,partitions,path)
        spark.sql(stmt)

    if not existingTableIsManaged:
        RemoveOldArchiveCopies(tableName,destination_filepath)

StatementMeta(, 4e9b6787-4649-41c6-979d-744279977fa4, 7, Finished, Available, Finished, False)

## Master for creating table

In [ ]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
    ('PCN',                       'int',            ''), 
    ('Price_Key',                 'int',            ''), 
    ('PO_Line_Key',               'int',            ''), 
    ('Effective_Date',            'timestamp',      ''), 
    ('Part_Status',               'string',         ''), 
    ('Price',                     'decimal(16,6)',  ''), 
    ('Active',                    'smallint',       ''), 
    ('Expiration_Date',           'timestamp',      ''), 
    ('Unit',                      'string',         ''), 
    ('Add_Date',                  'timestamp',      ''), 
    ('Update_Date',               'timestamp',      ''), 
    ('Cost',                      'decimal(16,6)',  ''), 
    ('Margin',                    'decimal(16,6)',  ''), 
    ('Part_Cost',                 'decimal(16,6)',  ''), 
    ('Price_In_Effect',           'boolean',        ''), 
    ('Price_In_Effect_Basis',     'decimal(16,6)',  ''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_SourceFileOrFolder',      'string',     ''), 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('AuditPipelineTriggerTime',        'timestamp',  ''),
    ('AuditPipelineRunId',        'string',  ''),
    ('AuditFilePath',        'string',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "test_plex"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


## Process exploded_bom_query

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('exploded_bomKey',         'binary',  ''),
('exploded_bomKeyInt',         'bigint',  ''),
('PCN','int',''),
('Version','string',''),
('cost_model_key','int',''),
('Child_Key','int',''),
('Part_Operation_Key','int',''),
('Parent_Part_Key','int',''),
('parent_part_no','varchar(500)',''),
('Description','varchar(4000)',''),
('Revision','varchar(50)',''),
('Name','varchar(4000)',''),
('Operation_No','int',''),
('Operation_Code','varchar(50)',''),
('Cost_Sub_Type_Key','int',''),
('Cost_Type','varchar(50)',''),
('Cost_Sub_Type','varchar(50)',''),
('Quantity','decimal(16,6)',''),
('Cost','decimal(16,6)',''),
('Level','int',''),
('Path','varchar(4000)',''),
('CT_Sort_Order','int',''),
('CST_Sort_Order','int',''),
('PartRowSpan','int',''),
('OpRowSpan','int',''),
('Note','varchar(4000)',''),
('Calc_Note','varchar(4000)',''),
('zero_level_desc','varchar(50)',''),

]

# Audit-prefixed columns to include for every table 
audit_columns = [  
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "exploded_bom_query"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, d0f171c3-8825-468b-84fd-653a94c125cb, 8, Finished, Available, Finished)

## Process Shipment

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('shipmentKey',         'binary',  ''),
('shipmentKeyInt',         'bigint',  ''),
('PCN','int',''),
('Customer_No','String',''),
('Source','String',''),
('Customer_Code','String',''),
('CustomerAddressKey','binary',''),
('Part_Key','int',''),
('part_masterKey',         'binary',  ''),
('Customer_Part_Key','binary',''),
('name','string',''),
('Ship_Date','timestamp',''),
('Part_No','string',''),
('Customer_Part_No','String',''),
('Quantity_Shipped','decimal(16,4)',''),
('price','decimal(16,4)',''),
##('Weight_Shipped','decimal(16,4)',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "shipment"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, a5bed4c5-0f03-4906-ae98-e190ba1c3a7d, 8, Finished, Available, Finished)

## Process VW_WRTariff

In [8]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('vw_wrtariffKey',         'binary',  ''),
('vw_wrtariffKeyInt',         'bigint',  ''),
('ProcessID','int',''),
('Year','int',''),
('machine_activity','decimal(28,9)',''),
('worker_activity','decimal(28,9)',''),
('Total_machine_activity','decimal(28,9)',''),
('Total_worker_activity','decimal(28,9)',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "vw_wrtariff"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, d0d45416-e388-41ea-a965-c7269121b7e6, 10, Finished, Available, Finished)

## Process vw_project_cost_details

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('VW_Project_cost_detailsKey',         'binary',  ''),
('VW_Project_cost_detailsKeyInt',         'bigint',  ''),  
('projectid','int',''),
('projectnumber','String',''),
('captiondefault','String',''),
('captionEN','String',''),
('projecttype','int',''),
('number_of_years','int',''),
('Quantity','int',''),
('SOP','timestamp',''),
('EOP','timestamp',''),
('Customer','String',''),
('Program','String',''),
('Commodity','String',''),
('REgion','String',''),
('Prod_family','String',''),
('Prod_group','String',''),
('partid','int',''),
('partnumber','String',''),
('PartDesc','String',''),
('parentid','int',''),
('currencyid','int',''),
('level','int',''),
('lininglevel','String',''),
('sort','int',''),
('parttype','String',''),
('relativequantity','decimal(28,9)',''),
('Year','int',''),
('TotalPurchasedParts','int',''),
('TotalIntercompanyPart','int',''),
('TotalMaterial','int',''),
('TotalMaterial_plus_IC_Profit','int',''),
('Total_Labor_Costs','int',''),
('COGS','int',''),
('PurchasedParts','int',''),
('FreightIn','int',''),
('ScrapPurchasedParts','int',''),
('Material_w_IC_Marukup_incl','int',''),
('FreightRawMaterial','int',''),
('TotalRawMaterial_wo_Scrap','int',''),
('TotalPP_wo_Scrap','int',''),
('IC_Material','int',''),
('IC_Scrap','int',''),
('IC_Profit','int',''),
('IC_Freight','int',''),
('TotalPlantCosts','int',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "vw_project_cost_details"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 9f4c5e03-3b93-4e10-a792-1fb8852c102b, 8, Finished, Available, Finished)

## Process vw_part_surcharge_details

In [7]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('vw_part_surcharge_detailsKey',         'binary',  ''),
('vw_part_surcharge_detailsKeyInt',         'bigint',  ''), 
('ProcessID','int',''),
('CaptionDefault','varchar(255)',''),
('partid','int',''),
('ProjectID','int',''),
('year','int',''),
('VariableBurden','decimal(28,9)',''),
('FixBurden','decimal(28,9)',''),
('BurdenCost','decimal(28,9)',''),
('MachineScrap','decimal(28,9)',''),
('MachineEfficiency','decimal(28,9)',''),
('LaborCost','decimal(28,9)',''),
('LaborWithInflation','decimal(28,9)',''),
('LaborScrap','decimal(28,9)',''),
('LaborEfficiency','decimal(28,9)',''),
('Crew','decimal(28,9)',''),
('Parts_Per_Cycle','decimal(28,9)',''),
('EFF','decimal(28,9)',''),
('Total_Labor_Costs','decimal(28,9)',''),
('CycleTime','decimal(28,9)',''),
('TotalVariable','decimal(28,9)',''),
('TotalBurden','decimal(28,9)',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "vw_part_surcharge_details"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 9f4c5e03-3b93-4e10-a792-1fb8852c102b, 9, Finished, Available, Finished)

## Process BOM_approved_workcenter

In [10]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('bom_approved_workcenterKey',         'binary',  ''),
('bom_approved_workcenterKeyInt',         'bigint',  ''),    
('Plexus_Customer_No','int',''),
#('Customer_part_key','int',''),
#('customer_No','double',''),
#('Customer_part_no','string',''),
#('Customer_part_description','string',''),
#('Customer_Code','string',''),
#('Customer_Name','string',''),
('Source','string',''),
#('customer_status','string',''),
#('customer_type','string',''),
#('PO_Line_Key','int',''),
#('PO_Key','int',''),
('Part_Key','int',''),
#('Effective_Date','timestamp',''),
#('expiration_date','timestamp',''),
#('unit','string',''),
#('price','decimal(20,4)',''),
('Part_Operation_Key','int',''),
('Operation_No','int',''),
('Operation_Key','int',''),
('Description','int',''),
('Crew_Size','int',''),
('Sort_Order','int',''),
('Target_Rate','decimal(10,6)',''),
('Workcenter_Key','int',''),
('Workcenter_Code','string',''),
('workcenterName','string',''),
('Variable_Cost','decimal(10,6)',''),
('Overhead_Cost','decimal(10,6)',''),
('Direct_Labor_Cost','decimal(10,6)',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "BOM_approved_workcenter"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 9f4c5e03-3b93-4e10-a792-1fb8852c102b, 12, Finished, Available, Finished)

## Process approved_workcenter

In [6]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('approved_workcenterKey',         'binary',  ''),
('approved_workcenterKeyInt',         'bigint',  ''),    
('Plexus_Customer_No','int',''),
('Customer_part_key','int',''),
('customer_No','double',''),
('Customer_part_no','string',''),
('Customer_part_description','string',''),
('Customer_Code','string',''),
('Customer_Name','string',''),
('Source','string',''),
('customer_status','string',''),
('customer_type','string',''),
('PO_Line_Key','int',''),
('PO_Key','int',''),
('Part_Key','int',''),
('Effective_Date','timestamp',''),
('expiration_date','timestamp',''),
('unit','string',''),
('price','decimal(20,4)',''),
('Part_Operation_Key','int',''),
('Operation_No','int',''),
('Operation_Key','int',''),
('Description','int',''),
('Crew_Size','int',''),
('Sort_Order','int',''),
('Target_Rate','decimal(10,6)',''),
('Workcenter_Key','int',''),
('Workcenter_Code','string',''),
('workcenterName','string',''),
('Variable_Cost','decimal(10,6)',''),
('Overhead_Cost','decimal(10,6)',''),
('Direct_Labor_Cost','decimal(10,6)',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "approved_workcenter"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, b26d8b36-4a7b-4d86-92ca-bc54dacc3a6e, 8, Finished, Available, Finished)

## Process exploded_bom

In [8]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('exploded_bomKey',         'binary',  ''),
('exploded_bomKeyInt',         'bigint',  ''),
('PCN','int',''),
('Version','string',''),
('cost_model_key','int',''),
('Child_Key','int',''),
('Part_Operation_Key','int',''),
('Parent_Part_Key','int',''),
('parent_part_no','varchar(500)',''),
('Description','varchar(4000)',''),
('Revision','varchar(50)',''),
('Name','varchar(4000)',''),
('Operation_No','int',''),
('Operation_Code','varchar(50)',''),
('Cost_Sub_Type_Key','int',''),
('Cost_Type','varchar(50)',''),
('Cost_Sub_Type','varchar(50)',''),
('Quantity','decimal(16,6)',''),
('Cost','decimal(16,6)',''),
('Level','int',''),
('Path','varchar(4000)',''),
('CT_Sort_Order','int',''),
('CST_Sort_Order','int',''),
('PartRowSpan','int',''),
('OpRowSpan','int',''),
('Note','varchar(4000)',''),
('Calc_Note','varchar(4000)',''),
('zero_level_desc','varchar(50)',''),

]

# Audit-prefixed columns to include for every table 
audit_columns = [  
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "exploded_bom"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 2de1e821-f90d-46a3-8946-cc4590b6453b, 10, Finished, Available, Finished)

## Procees parts_master

In [14]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('part_masterKey',         'binary',  ''),
('part_masterKeyInt',         'bigint',  ''),
('PCN','int',''),
('Part_Key','int',''),
('Part_No','string',''),
('name','string',''),
('revision','string',''),
('Part_type','string',''),
('Part_status','string',''),
('unit','string',''),
('building_key','int',''),
('Part_group_key','int',''),
('Part_Group','string',''),
('Product_type_key','int',''),
('Product_type','string',''),
('Product_type_Code','string',''),
('Part_source_key','int',''),
('Purchased_Part','boolean',''),
('Manufactured_Part','boolean',''),
('Source','String',''),
]

# Audit-prefixed columns to include for every table 
audit_columns = [  
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "part_master"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, 458f8e6f-9ecc-491b-b64e-c587e643d33c, 16, Finished, Available, Finished)

#### Process to create table QAD_Part_Attribute

In [8]:
base_columns = [
 ('QAD_Part_Attribute_Key',         'binary','')
,('QAD_Part_Attribute_KeyInt',         'bigint','')
,('Site','string','')
,('PCN','string','')
,('Part','string','')
,('Description','string','')
,('Type','string','')
,('Group','string','')
,('Product_Line','string','')
,('Product_Line_Description','string','')
,('Routing','string','')
,('Work_Center','string','')
,('Operation','string','')
,('Unit','string','')
,('PM_Code','string','')
]

# Audit-prefixed columns to include for every table 
audit_columns = [  
    ('Audit_CreatedDateTime',         'timestamp',  ''), 
    ('Audit_ModifiedDateTime',        'timestamp',  ''),
    ('Audit_Source_ModifiedDateTime',        'timestamp',  ''),
    ('RevisionHash',        'binary',  ''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "QAD_Part_Attribute"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)

StatementMeta(, b8367eee-f0f0-4cd2-b968-a079e85ca033, 10, Finished, Available, Finished)

#### Process table Commodity.

In [6]:
comparisonArray=[
	{"col_name":"CommodityKey", "data_type":"binary","comment":"[ID:{84947}]"}
	,{"col_name":"CommodityKeyInt", "data_type":"bigint","comment":"[ID:{84947INT}]"}
	,{"col_name":"PartNumber", "data_type":"string","comment":"[ID:{67469}]"}
	,{"col_name":"PlexPCN", "data_type":"string","comment":"[ID:{49948}]"}
	,{"col_name":"Commodity", "data_type":"string","comment":"[ID:{11613}]"}
	,{"col_name":"SubCommodity", "data_type":"string","comment":"[ID:{59576}]"}
	,{"col_name":"MainCommodityShort", "data_type":"string","comment":"[ID:{14413}]"}
	,{"col_name":"MainCommodityLong", "data_type":"string","comment":"[ID:{24554}]"}
	,{"col_name":"MaterialType", "data_type":"string","comment":"[ID:{46777}]"}
	,{"col_name":"QualityRelevant", "data_type":"string","comment":"[ID:{41278}]"}
	,{"col_name":"PartType", "data_type":"string","comment":"[ID:{11501}]"}
	,{"col_name":"MaterialContent", "data_type":"string","comment":"[ID:{74839}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Commodity"

ProcessTable("LH_BI_SILVER_TIER2","Commodity",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 8, Finished, Available, Finished)

#### Process table FiscalPeriod.

In [7]:
comparisonArray=[
	{"col_name":"FiscalPeriodKey", "data_type":"binary","comment":"[ID:{52664}]"}
	,{"col_name":"FiscalPeriodKeyInt", "data_type":"bigint","comment":"[ID:{52664INT}]"}
	,{"col_name":"PCN", "data_type":"string","comment":"[ID:{74040}]"}
	,{"col_name":"Period", "data_type":"string","comment":"[ID:{57987}]"}
	,{"col_name":"PeriodStatus", "data_type":"string","comment":"[ID:{86536}]"}
	,{"col_name":"Display", "data_type":"string","comment":"[ID:{87046}]"}
	,{"col_name":"BeginDate", "data_type":"date","comment":"[ID:{93240}]"}
	,{"col_name":"EndDate", "data_type":"date","comment":"[ID:{90944}]"}
	,{"col_name":"PeriodDisplay", "data_type":"smallint","comment":"[ID:{42645}]"}
	,{"col_name":"FiscalOrder", "data_type":"smallint","comment":"[ID:{52547}]"}
	,{"col_name":"PeriodKey", "data_type":"string","comment":"[ID:{14380}]"}
	,{"col_name":"QuarterGroup", "data_type":"smallint","comment":"[ID:{40503}]"}
	,{"col_name":"AddDate", "data_type":"string","comment":"[ID:{26161}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"FiscalPeriod"

ProcessTable("LH_BI_SILVER_TIER2","FiscalPeriod",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 9, Finished, Available, Finished)

#### Process table Part.

In [8]:
comparisonArray=[
	{"col_name":"PartKey", "data_type":"binary","comment":"[ID:{46968}]"}
	,{"col_name":"PartKeyInt", "data_type":"bigint","comment":"[ID:{46968INT}]"}
	,{"col_name":"PartNumber", "data_type":"string","comment":"[ID:{74879}]"}
	,{"col_name":"Site", "data_type":"string","comment":"[ID:{49872}]"}
	,{"col_name":"Source", "data_type":"string","comment":"[ID:{15342}]"}
	,{"col_name":"Name", "data_type":"string","comment":"[ID:{79605}]"}
	,{"col_name":"Unit", "data_type":"string","comment":"[ID:{28792}]"}
	,{"col_name":"PlexPartKey", "data_type":"int","comment":"[ID:{34272}]"}
	,{"col_name":"PlexPCN", "data_type":"string","comment":"[ID:{36439}]"}
	,{"col_name":"PartType", "data_type":"string","comment":"[ID:{85095}]"}
	,{"col_name":"PartStatus", "data_type":"string","comment":"[ID:{63367}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_IsMissingPrimaryKey", "data_type":"boolean","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Part"

ProcessTable("LH_BI_SILVER_TIER2","Part",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 10, Finished, Available, Finished)

#### Process table Plant.

In [9]:
comparisonArray=[
	{"col_name":"PlantKey", "data_type":"binary","comment":"[ID:{45216}]"}
	,{"col_name":"PlantKeyInt", "data_type":"bigint","comment":"[ID:{45216INT}]"}
	,{"col_name":"PlantName", "data_type":"string","comment":"[ID:{11253}]"}
	,{"col_name":"ERPSite", "data_type":"string","comment":"[ID:{10713}]"}
	,{"col_name":"QADSite", "data_type":"string","comment":"[ID:{45304}]"}
	,{"col_name":"PCN", "data_type":"string","comment":"[ID:{60933}]"}
	,{"col_name":"PlexName", "data_type":"string","comment":"[ID:{93120}]"}
	,{"col_name":"ShortName", "data_type":"string","comment":"[ID:{67632}]"}
	,{"col_name":"Region", "data_type":"string","comment":"[ID:{68587}]"}
	,{"col_name":"ERP", "data_type":"string","comment":"[ID:{80029}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Plant"

ProcessTable("LH_BI_SILVER_TIER2","Plant",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 11, Finished, Available, Finished)

#### Process table Receipt.

In [10]:
comparisonArray=[
	{"col_name":"ReceiptKey", "data_type":"binary","comment":"[ID:{62272}]"}
	,{"col_name":"ReceiptKeyInt", "data_type":"bigint","comment":"[ID:{62272INT}]"}
	,{"col_name":"QADKey", "data_type":"string","comment":"[ID:{88499}]"}
	,{"col_name":"PlexPCN", "data_type":"string","comment":"[ID:{21903}]"}
	,{"col_name":"PlexReceiptKey", "data_type":"bigint","comment":"[ID:{58150}]"}
	,{"col_name":"PlexPartKey", "data_type":"bigint","comment":"[ID:{19592}]"}
	,{"col_name":"PartNumber", "data_type":"string","comment":"[ID:{16458}]"}
	,{"col_name":"LineItemNo", "data_type":"string","comment":"[ID:{97018}]"}
	,{"col_name":"SupplierNo", "data_type":"string","comment":"[ID:{73266}]"}
	,{"col_name":"Receiver", "data_type":"string","comment":"[ID:{90295}]"}
	,{"col_name":"Site", "data_type":"string","comment":"[ID:{10928}]"}
	,{"col_name":"Unit", "data_type":"string","comment":"[ID:{82013}]"}
	,{"col_name":"ReceivedDate", "data_type":"date","comment":"[ID:{92992}]"}
	,{"col_name":"ReceivedQuantity", "data_type":"double","comment":"[ID:{34938}]"}
	,{"col_name":"Source", "data_type":"string","comment":"[ID:{40364}]"}
	,{"col_name":"Currency", "data_type":"string","comment":"[ID:{46258}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Receipt"

ProcessTable("LH_BI_SILVER_TIER2","Receipt",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 12, Finished, Available, Finished)

#### Process table Supplier.

In [11]:
comparisonArray=[
	{"col_name":"SupplierKey", "data_type":"binary","comment":"[ID:{61043}]"}
	,{"col_name":"SupplierKeyInt", "data_type":"bigint","comment":"[ID:{61043INT}]"}
	,{"col_name":"PlexPCN", "data_type":"string","comment":"[ID:{45494}]"}
	,{"col_name":"Site", "data_type":"string","comment":"[ID:{75694}]"}
	,{"col_name":"Currency", "data_type":"string","comment":"[ID:{59439}]"}
	,{"col_name":"PlexSupplierNo", "data_type":"string","comment":"[ID:{15826}]"}
	,{"col_name":"LegacySupplier", "data_type":"string","comment":"[ID:{76558}]"}
	,{"col_name":"SupplierName", "data_type":"string","comment":"[ID:{23944}]"}
	,{"col_name":"MultiPlantSupplierName", "data_type":"string","comment":"[ID:{56950}]"}
	,{"col_name":"SupplierType", "data_type":"string","comment":"[ID:{74746}]"}
	,{"col_name":"PlexSupplierCode", "data_type":"string","comment":"[ID:{15076}]"}
	,{"col_name":"Source", "data_type":"string","comment":"[ID:{15116}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Supplier"

ProcessTable("LH_BI_SILVER_TIER2","Supplier",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 13, Finished, Available, Finished)

#### Process table Customer Address

In [12]:
comparisonArray=[
	{"col_name":"CustomerAddressKey", "data_type":"binary","comment":"[ID:{61043}]"}
	,{"col_name":"CustomerAddressKeyInt", "data_type":"bigint","comment":"[ID:{61043INT}]"}
	,{"col_name":"PCN", "data_type":"int","comment":"[ID:{29615}]"}
	,{"col_name":"Customer_No", "data_type":"string","comment":"[ID:{84932}]"}
	,{"col_name":"Customer_Code", "data_type":"string","comment":"[ID:{84933}]"}
	,{"col_name":"Name", "data_type":"string","comment":"[ID:{84934}]"}
	,{"col_name":"Customer_Status", "data_type":"string","comment":"[ID:{84935}]"}
	,{"col_name":"Customer_Type", "data_type":"string","comment":"[ID:{84936}]"}
	,{"col_name":"Customer_Address_No", "data_type":"string","comment":"[ID:{73045}]"}
	,{"col_name":"Customer_Address_Code", "data_type":"string","comment":"[ID:{44383}]"}
	,{"col_name":"Address", "data_type":"string","comment":"[ID:{50616}]"}
    ,{"col_name":"City", "data_type":"string","comment":"[ID:{50617}]"}
    ,{"col_name":"State", "data_type":"string","comment":"[ID:{50618}]"}
    ,{"col_name":"Zip", "data_type":"string","comment":"[ID:{50619}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"customer_address"

ProcessTable("LH_BI_SILVER_TIER2","customer_address",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 14, Finished, Available, Finished)

#### Process table Sales

In [7]:
comparisonArray=[
    {"col_name":"SalesKey", "data_type":"binary","comment":"[ID:{64416}]"}
    ,{"col_name":"SalesKeyInt", "data_type":"bigint","comment":"[ID:{75429}]"}
    ,{"col_name":"System", "data_type":"string","comment":"[ID:{62277}]"}
    ,{"col_name":"PlexPCN", "data_type":"int","comment":"[ID:{87245}]"}
    ,{"col_name":"QADSite", "data_type":"string","comment":"[ID:{73314}]"}
    ,{"col_name":"Invoice_No", "data_type":"string","comment":"[ID:{12189}]"}
    ,{"col_name":"BillTo", "data_type":"string","comment":"[ID:{94075}]"}
    ,{"col_name":"ShipTo", "data_type":"string","comment":"[ID:{30756}]"}
    ,{"col_name":"Invoice_Date", "data_type":"date","comment":"[ID:{71501}]"}
    ,{"col_name":"Invoice_Line", "data_type":"string","comment":"[ID:{97531}]"}
    ,{"col_name":"Part", "data_type":"string","comment":"[ID:{97532}]"}
    ,{"col_name":"Quantity", "data_type":"string","comment":"[ID:{48325}]"}
    ,{"col_name":"Price", "data_type":"string","comment":"[ID:{65544}]"}
    ,{"col_name":"Customer_PO", "data_type":"string","comment":"[ID:{45375}]"}
    ,{"col_name":"Account", "data_type":"string","comment":"[ID:{86550}]"}
    ,{"col_name":"Currency", "data_type":"string","comment":"[ID:{25195}]"}
    ,{"col_name":"Customer_Part", "data_type":"string","comment":"[ID:{24851}]"}
    ,{"col_name":"Note", "data_type":"string","comment":"[ID:{69529}]"}
    ,{"col_name":"Sales_Order", "data_type":"string","comment":"[ID:{55553}]"}
    ,{"col_name":"Supplier", "data_type":"string","comment":"[ID:{14877}]"}
    ,{"col_name":"Invoice_Amount", "data_type":"string","comment":"[ID:{14878}]"}
    ,{"col_name":"Period", "data_type":"string","comment":"[ID:{14879}]"}
    ,{"col_name":"Exchange_rate", "data_type":"string","comment":"[ID:{14880}]"}
    ,{"col_name":"Currency_amount", "data_type":"string","comment":"[ID:{14881}]"}
    ,{"col_name":"Invoice_Link", "data_type":"string","comment":"[ID:{14882}]"}
    ,{"col_name":"Bill_to_Address", "data_type":"string","comment":"[ID:{14883}]"}#changed to string due to error
    ,{"col_name":"Ship_to_Address", "data_type":"string","comment":"[ID:{14884}]"} #changed to string due to error
    ,{"col_name":"Ship_date", "data_type":"string","comment":"[ID:{14885}]"}
    ,{"col_name":"Shipper_No", "data_type":"string","comment":"[ID:{14886}]"} #changed to string due to error
    ,{"col_name":"Supplier_No", "data_type":"string","comment":"[ID:{14887}]"} #changed to string due to error
    ,{"col_name":"Customer_No", "data_type":"string","comment":"[ID:{14888}]"} #changed to string due to error
    ,{"col_name":"PO_No", "data_type":"string","comment":"[ID:{14889}]"} #changed to string due to error
    ,{"col_name":"part_key", "data_type":"int","comment":"[ID:{14890}]"}
	,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"Sales"
    
ProcessTable("LH_BI_SILVER_TIER2","Sales",comparisonArray,partitions,path)

StatementMeta(, 4e9b6787-4649-41c6-979d-744279977fa4, 9, Finished, Available, Finished, False)

#### Process table Currency

In [14]:
comparisonArray=[
    {"col_name":"currencykey", "data_type":"binary","comment":"[ID:{57179}]"}
    ,{"col_name":"currencykeyint", "data_type":"bigint","comment":"[ID:{64342}]"}
    ,{"col_name":"plexcurrencykey", "data_type":"int","comment":"[ID:{5883}]"}
    ,{"col_name":"currencysymbol", "data_type":"string","comment":"[ID:{59120}]"}
    ,{"col_name":"currencyhtml", "data_type":"string","comment":"[ID:{93631}]"}
    ,{"col_name":"exchangerate", "data_type":"double","comment":"[ID:{85603}]"}
    ,{"col_name":"exchangeratedate", "data_type":"date","comment":"[ID:{91703}]"}
    ,{"col_name":"currencycode", "data_type":"string","comment":"[ID:{56237}]"}
    ,{"col_name":"description", "data_type":"string","comment":"[ID:{24726}]"}
    ,{"col_name":"sortorder", "data_type":"int","comment":"[ID:{82746}]"}
    ,{"col_name":"updatedate", "data_type":"date","comment":"[ID:{93344}]"}
    ,{"col_name":"active", "data_type":"boolean","comment":"[ID:{72790}]"}
    ,{"col_name":"currencyunicode", "data_type":"string","comment":"[ID:{96939}]"}
    ,{"col_name":"Audit_Source_ModifiedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"RevisionHash", "data_type":"binary","comment":""}
	,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=[]
path=destination_filepath+"currency"

ProcessTable("LH_BI_SILVER_TIER2","currency",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 16, Finished, Available, Finished)

#### Process to Create QAD_Shipments

In [7]:


# Provide Table-specific columns: (name, data_type, comment)
base_columns = [
('QADShipmentKey','binary',''),
('QADShipmentKeyInt','bigint',''),
('PCN','string',''),
('Tran_No','String',''),
('Line','int',''),
('Part_No','String',''),
('Quantity','decimal(16,4)',''),
('Customer_Code','string',''),
('Customer_Name','string',''),
('Ship_Date','timestamp',''),
('Customer_Part','string',''),
('Price','decimal(16,4)',''),
('Lot','String',''),
('Tran_Seq','int',''),
('System','string',''),
('Prod_Line','string',''),
('Prod_Line_Desc','string','')
]

# Audit-prefixed columns to include for every table 
audit_columns = [ 
    ('Audit_CreatedDateTime','timestamp',''), 
    ('Audit_ModifiedDateTime','timestamp',''),
    ('Audit_Source_ModifiedDateTime','timestamp',''),
    ('RevisionHash','binary',''),
    ]

# Combine base and audit columns
all_columns = base_columns + audit_columns


comparisonArray = [
    {
        "col_name":    col,
        "data_type":   dt,
        "comment":     comment
    }
    for col, dt, comment in all_columns
]

partitions = []

# Table alias and path
table_alias = "QAD_shipments"
path = destination_filepath + table_alias

# Invoke ProcessTable
ProcessTable(
    "LH_BI_SILVER_TIER2",
    table_alias,
    comparisonArray,
    partitions,
    path
)


StatementMeta(, bfb82584-cacd-4d4f-b618-3c75bf2666f6, 9, Finished, Available, Finished)

#### Process table Audit_RuntimeTableLog.

In [15]:
comparisonArray=[
	{"col_name":"GroupingID", "data_type":"string","comment":""}
    ,{"col_name":"StartDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"DatabaseName", "data_type":"string","comment":""}
    ,{"col_name":"TableName", "data_type":"string","comment":""}
    ,{"col_name":"TableStartTime", "data_type":"timestamp","comment":""}
    ,{"col_name":"TableEndTime", "data_type":"timestamp","comment":""}
    ,{"col_name":"TableDurationSeconds", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsAffected", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsInserted", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsUpdated", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsDeleted", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsBefore", "data_type":"bigint","comment":""}
    ,{"col_name":"RowsAfter", "data_type":"bigint","comment":""}
    ,{"col_name":"Audit_Status", "data_type":"string","comment":""}
    ,{"col_name":"Audit_CreatedDateTime", "data_type":"timestamp","comment":""}
	,{"col_name":"Audit_ModifiedDateTime", "data_type":"timestamp","comment":""}
]
partitions=["StartDateTime","DatabaseName","TableName"]

path=destination_filepath+"Audit_RuntimeTableLog"

ProcessTable("LH_BI_SILVER_TIER2","Audit_RuntimeTableLog",comparisonArray,partitions,path)

StatementMeta(, 5751fb0a-e27b-4e76-b4d8-3ed5408b975f, 17, Finished, Available, Finished)

#### End of Notebook